# SemEval-2018 Task 1 E-c: Improved Multi-Label Emotion Classification

This notebook is a **complete Colab-ready training pipeline** for the English **SemEval-2018 Task 1 E-c** dataset.

## What this notebook does
- downloads the dataset from Kaggle via `kagglehub`
- trains a **DeBERTa-v3-base** multi-label emotion classifier
- handles **11 emotions**
- uses **weighted BCEWithLogitsLoss** for label imbalance
- uses **AdamW**, scheduler, and **early stopping**
- saves the **best model**
- tunes **per-label thresholds** on the dev set
- reports **macro-F1**, **micro-F1**, precision, recall, and per-label F1
- exports validation and optional test predictions

## Labels
The SemEval-2018 E-c English task uses these 11 labels:

`anger, anticipation, disgust, fear, joy, love, optimism, pessimism, sadness, surprise, trust`

## Runtime
In Colab, use:

**Runtime -> Change runtime type -> T4 GPU**

## 1. Install dependencies
Run this cell once. In Colab it may take a minute.

In [40]:
!pip -q install -U transformers datasets accelerate scikit-learn kagglehub iterative-stratification sentencepiece

In [41]:
!pip install -U transformers accelerate

## 2. Imports and configuration
You can change the model or batch size here.

Notes:
- `microsoft/deberta-v3-base` is a strong default for English.
- If memory gets tight, reduce `MAX_LEN` to 96 and `TRAIN_BATCH_SIZE` to 8.

In [42]:
import os
import random
import math
import json
import gc
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed,
)
import kagglehub

In [43]:
# ===== CONFIG =====
MODEL_NAME = "distilbert-base-uncased"
LEARNING_RATE = 2e-5
EPOCHS = 5
MAX_LEN = 128
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 32

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2
SEED = 42

TEXT_COLUMN = "text"
ID_COLUMN = "ID"

LABEL_COLUMNS = [
    "anger", "anticipation", "disgust", "fear", "joy", "love",
    "optimism", "pessimism", "sadness", "surprise", "trust"
]

OUTPUT_DIR = "/content/emotion_model_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
USE_FP16 = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Device: cuda
GPU: Tesla T4


## 3. Download and load the SemEval-2018 E-c English dataset
This uses the Kaggle mirror instead of local files.

In [44]:
path = kagglehub.dataset_download("context/semeval-2018-task-ec")
print("Downloaded dataset path:", path)

train_path = os.path.join(path, "2018-E-c-En-train.txt")
dev_path = os.path.join(path, "2018-E-c-En-dev.txt")

train_df = pd.read_csv(train_path, sep="\t")
dev_df = pd.read_csv(dev_path, sep="\t")

# Standardize the text column name
train_df = train_df.rename(columns={"Tweet": TEXT_COLUMN})
dev_df = dev_df.rename(columns={"Tweet": TEXT_COLUMN})

# Basic checks
required_cols = [ID_COLUMN, TEXT_COLUMN] + LABEL_COLUMNS
missing_train = [c for c in required_cols if c not in train_df.columns]
missing_dev = [c for c in required_cols if c not in dev_df.columns]
assert not missing_train, f"Missing columns in train_df: {missing_train}"
assert not missing_dev, f"Missing columns in dev_df: {missing_dev}"

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
display(train_df[[ID_COLUMN, TEXT_COLUMN] + LABEL_COLUMNS].head())

Using Colab cache for faster access to the 'semeval-2018-task-ec' dataset.
Downloaded dataset path: /kaggle/input/semeval-2018-task-ec
Train shape: (6838, 13)
Dev shape: (886, 13)


,ID,text,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2017-En-21441,“Worry is a down payment on a problem you may ...,0,1,0,0,0,0,1,0,0,0,1
1,2017-En-31535,Whatever you decide to do make sure it makes y...,0,0,0,0,1,1,1,0,0,0,0
2,2017-En-21068,@Max_Kellerman it also helps that the majorit...,1,0,1,0,1,0,1,0,0,0,0
3,2017-En-31436,Accept the challenges so that you can literall...,0,0,0,0,1,0,1,0,0,0,0
4,2017-En-22195,My roommate: it's okay that we can't spell bec...,1,0,1,0,0,0,0,0,0,0,0


## 4. Quick dataset inspection

In [45]:
print("Training samples:", len(train_df))
print("Validation samples:", len(dev_df))
print("\nEmotion distribution in training data:")
display(train_df[LABEL_COLUMNS].sum().sort_values(ascending=False).to_frame("count"))

Training samples: 6838
Validation samples: 886

Emotion distribution in training data:


,count
disgust,2602
anger,2544
joy,2477
sadness,2008
optimism,1984
fear,1242
anticipation,978
pessimism,795
love,700
surprise,361


## 5. Tokenizer

In [46]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LEN,
    )

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 6. Convert pandas to Hugging Face datasets

In [47]:
for col in LABEL_COLUMNS:
    train_df[col] = train_df[col].astype(np.float32)
    dev_df[col] = dev_df[col].astype(np.float32)

def df_to_hf_dataset(df: pd.DataFrame) -> Dataset:
    df = df.copy()

    # ensure float labels
    for col in LABEL_COLUMNS:
        df[col] = df[col].astype(np.float32)

    # create labels column
    df["labels"] = df[LABEL_COLUMNS].values.astype(np.float32).tolist()

    ds = Dataset.from_pandas(
        df[[TEXT_COLUMN, "labels"]],
        preserve_index=False
    )

    def preprocess(batch):
        enc = tokenizer(
            batch[TEXT_COLUMN],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
        )

        # IMPORTANT: assign labels AFTER tokenization
        enc["labels"] = batch["labels"]
        return enc

    ds = ds.map(preprocess, batched=True)

    # CRITICAL: remove all unused columns
    ds = ds.remove_columns([TEXT_COLUMN])

    cols = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in ds.column_names:
        cols.append("token_type_ids")

    ds.set_format(type="torch", columns=cols)

    return ds

In [48]:
train_ds = df_to_hf_dataset(train_df)
dev_ds = df_to_hf_dataset(dev_df)

Map:   0%|          | 0/6838 [00:00<?, ? examples/s]

Map:   0%|          | 0/886 [00:00<?, ? examples/s]

In [49]:
print(train_ds[0])

{'labels': tensor([0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 1.]), 'input_ids': tensor([  101,  1523,  4737,  2003,  1037,  2091,  7909,  2006,  1037,  3291,
         2017,  2089,  2196,  2031,  1005,  1012, 11830, 11527,  1012,  1001,
        14354,  1001,  4105,  1001,  4737,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

## 7. Class weights for imbalance
We compute a positive weight for each label and use it inside `BCEWithLogitsLoss`.

In [50]:
label_matrix = train_df[LABEL_COLUMNS].values.astype(np.float32)
positive_counts = label_matrix.sum(axis=0)
negative_counts = len(train_df) - positive_counts

# Standard pos_weight for BCEWithLogitsLoss
pos_weight = torch.tensor(negative_counts / np.clip(positive_counts, 1, None), dtype=torch.float32)
print("Positive counts by label:")
for label, count, pw in zip(LABEL_COLUMNS, positive_counts.astype(int), pos_weight.tolist()):
    print(f"{label:12s} positives={count:4d}  pos_weight={pw:.3f}")

Positive counts by label:
anger        positives=2544  pos_weight=1.688
anticipation positives= 978  pos_weight=5.992
disgust      positives=2602  pos_weight=1.628
fear         positives=1242  pos_weight=4.506
joy          positives=2477  pos_weight=1.761
love         positives= 700  pos_weight=8.769
optimism     positives=1984  pos_weight=2.447
pessimism    positives= 795  pos_weight=7.601
sadness      positives=2008  pos_weight=2.405
surprise     positives= 361  pos_weight=17.942
trust        positives= 357  pos_weight=18.154


## 8. Model and custom trainer
We use a Hugging Face sequence classification model with:
- `problem_type="multi_label_classification"`
- custom weighted BCE loss

In [51]:
num_labels = len(LABEL_COLUMNS)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
)



model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 9. Metrics
During training, evaluation uses a default threshold of 0.5.  
After training, we do a separate **per-label threshold tuning** step on the dev set.

In [52]:
def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    logits = np.array(logits, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)

    logits = np.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0)
    labels = np.nan_to_num(labels, nan=0.0)

    probs = sigmoid(logits)
    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_precision = precision_score(labels, preds, average="macro", zero_division=0)
    macro_recall = recall_score(labels, preds, average="macro", zero_division=0)
    micro_precision = precision_score(labels, preds, average="micro", zero_division=0)
    micro_recall = recall_score(labels, preds, average="micro", zero_division=0)

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
    }

## 10. Training arguments
`load_best_model_at_end=True` keeps the checkpoint with the best validation macro-F1.

In [53]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLUMNS),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [54]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    do_train=True,
    do_eval=True,
    learning_rate=3e-6,   # 👈 reduced
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=VALID_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=50,
    fp16=False,
    seed=SEED,
    max_grad_norm=0.5   # 👈 stronger clipping
)

## 11. Train

In [55]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [56]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 12. Evaluate with default threshold = 0.5

In [57]:
print(type(train_ds), type(dev_ds))

<class 'datasets.arrow_dataset.Dataset'> <class 'datasets.arrow_dataset.Dataset'>


In [58]:
print("Train label NaNs:")
print(train_df[LABEL_COLUMNS].isna().sum())

print("\nDev label NaNs:")
print(dev_df[LABEL_COLUMNS].isna().sum())

Train label NaNs:
anger           0
anticipation    0
disgust         0
fear            0
joy             0
love            0
optimism        0
pessimism       0
sadness         0
surprise        0
trust           0
dtype: int64

Dev label NaNs:
anger           0
anticipation    0
disgust         0
fear            0
joy             0
love            0
optimism        0
pessimism       0
sadness         0
surprise        0
trust           0
dtype: int64


In [59]:
for col in LABEL_COLUMNS:
    print(col, sorted(train_df[col].dropna().unique())[:10], sorted(dev_df[col].dropna().unique())[:10])

anger [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
anticipation [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
disgust [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
fear [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
joy [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
love [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
optimism [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
pessimism [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
sadness [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
surprise [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]
trust [np.float32(0.0), np.float32(1.0)] [np.float32(0.0), np.float32(1.0)]


In [60]:
print(train_ds[0])
print(dev_ds[0])

{'labels': tensor([0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 1.]), 'input_ids': tensor([  101,  1523,  4737,  2003,  1037,  2091,  7909,  2006,  1037,  3291,
         2017,  2089,  2196,  2031,  1005,  1012, 11830, 11527,  1012,  1001,
        14354,  1001,  4105,  1001,  4737,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [61]:
print(trainer.state.log_history[:10])

[]


In [ ]:
trainer.train()

Step,Training Loss
50,0.665503
100,0.592198
150,0.533142
200,0.509633
250,0.495384
300,0.484643
350,0.472859
400,0.475243
450,0.473688
500,0.457150


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
default_metrics = trainer.evaluate(dev_ds)
print(default_metrics)

In [ ]:
default_metrics = trainer.evaluate(dev_ds)
print(json.dumps(default_metrics, indent=2))

In [ ]:
pred_output = trainer.predict(dev_ds)

In [ ]:
probs = sigmoid(pred_output.predictions)
print(probs[:3])

## 13. Tune per-label thresholds on the dev set
This often improves macro-F1 for imbalanced multi-label tasks.

In [ ]:
pred_output = trainer.predict(dev_ds)
dev_logits = pred_output.predictions
dev_labels = pred_output.label_ids.astype(int)
dev_probs = sigmoid(dev_logits)

threshold_grid = np.arange(0.05, 0.5, 0.02)

best_thresholds = []
best_label_f1 = []

for i, label in enumerate(LABEL_COLUMNS):
    y_true = dev_labels[:, i]
    best_thr = 0.5
    best_f1 = -1.0

    for thr in threshold_grid:
        y_pred = (dev_probs[:, i] >= thr).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thr = float(thr)

    best_thresholds.append(best_thr)
    best_label_f1.append(best_f1)

threshold_df = pd.DataFrame({
    "label": LABEL_COLUMNS,
    "best_threshold": best_thresholds,
    "best_dev_f1": best_label_f1
}).sort_values("best_dev_f1", ascending=False)

display(threshold_df)

## 14. Final dev metrics with tuned thresholds

In [ ]:
best_thresholds_arr = np.array(best_thresholds).reshape(1, -1)
dev_preds_tuned = (dev_probs >= best_thresholds_arr).astype(int)

macro_f1_tuned = f1_score(dev_labels, dev_preds_tuned, average="macro", zero_division=0)
micro_f1_tuned = f1_score(dev_labels, dev_preds_tuned, average="micro", zero_division=0)
macro_precision_tuned = precision_score(dev_labels, dev_preds_tuned, average="macro", zero_division=0)
macro_recall_tuned = recall_score(dev_labels, dev_preds_tuned, average="macro", zero_division=0)
micro_precision_tuned = precision_score(dev_labels, dev_preds_tuned, average="micro", zero_division=0)
micro_recall_tuned = recall_score(dev_labels, dev_preds_tuned, average="micro", zero_division=0)

tuned_metrics = {
    "macro_f1_tuned": macro_f1_tuned,
    "micro_f1_tuned": micro_f1_tuned,
    "macro_precision_tuned": macro_precision_tuned,
    "macro_recall_tuned": macro_recall_tuned,
    "micro_precision_tuned": micro_precision_tuned,
    "micro_recall_tuned": micro_recall_tuned,
}

print(json.dumps(tuned_metrics, indent=2))

## 15. Per-label classification report on dev

In [ ]:
report = classification_report(
    dev_labels,
    dev_preds_tuned,
    target_names=LABEL_COLUMNS,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()
display(report_df)

## 16. Save dev predictions, thresholds, and metrics

In [ ]:
# Save thresholds
with open(os.path.join(OUTPUT_DIR, "best_thresholds.json"), "w") as f:
    json.dump({label: thr for label, thr in zip(LABEL_COLUMNS, best_thresholds)}, f, indent=2)

# Save metrics
with open(os.path.join(OUTPUT_DIR, "dev_metrics_default.json"), "w") as f:
    json.dump(default_metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "dev_metrics_tuned.json"), "w") as f:
    json.dump(tuned_metrics, f, indent=2)

# Save dev predictions
dev_pred_df = dev_df[[ID_COLUMN, TEXT_COLUMN] + LABEL_COLUMNS].copy()
for i, label in enumerate(LABEL_COLUMNS):
    dev_pred_df[f"{label}_prob"] = dev_probs[:, i]
    dev_pred_df[f"{label}_pred"] = dev_preds_tuned[:, i]

dev_pred_path = os.path.join(OUTPUT_DIR, "dev_predictions_with_probs.csv")
dev_pred_df.to_csv(dev_pred_path, index=False)

print("Saved files to:", OUTPUT_DIR)
print("Dev predictions:", dev_pred_path)

## 17. Optional: Predict on a test file
If you have an external English test file with a `Tweet` column or a `text` column, upload it to Colab and set the path below.

This section is optional and safe to skip.

In [ ]:
TEST_FILE_PATH = "/content/sample_emotion_test.csv"

if TEST_FILE_PATH is not None and os.path.exists(TEST_FILE_PATH):

    test_df = pd.read_csv(TEST_FILE_PATH)

    if "Tweet" in test_df.columns and TEXT_COLUMN not in test_df.columns:
        test_df = test_df.rename(columns={"Tweet": TEXT_COLUMN})

    if ID_COLUMN not in test_df.columns:
        test_df[ID_COLUMN] = np.arange(len(test_df))

    assert TEXT_COLUMN in test_df.columns, f"Expected a '{TEXT_COLUMN}' column."

    test_ds = Dataset.from_pandas(
        test_df[[ID_COLUMN, TEXT_COLUMN]],
        preserve_index=False
    )

    test_ds = test_ds.map(tokenize_function, batched=True)

    # remove text/id columns before prediction
    remove_cols = [col for col in [ID_COLUMN, TEXT_COLUMN] if col in test_ds.column_names]
    test_ds = test_ds.remove_columns(remove_cols)

    # remove token_type_ids if present
    if "token_type_ids" in test_ds.column_names:
        test_ds = test_ds.remove_columns(["token_type_ids"])

    test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    test_output = trainer.predict(test_ds)
    test_probs = sigmoid(test_output.predictions)

    submission = test_df[[ID_COLUMN, TEXT_COLUMN]].copy()

    for i, label in enumerate(LABEL_COLUMNS):
        submission[f"{label}_prob"] = test_probs[:, i]

    test_out_path = os.path.join(OUTPUT_DIR, "test_probabilities.csv")
    submission.to_csv(test_out_path, index=False)

    print("Saved probability predictions to:", test_out_path)
    display(submission)

else:
    print("Skipped test prediction block.")

## 18. Quick inference on custom text
Use this to test the model on your own sentence.

In [ ]:
id2label = {i: label for i, label in enumerate(LABEL_COLUMNS)}

def predict_emotions(text):
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    enc = {k: v.to(trainer.model.device) for k, v in enc.items()}

    trainer.model.eval()
    with torch.no_grad():
        logits = trainer.model(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    result = pd.DataFrame({
        "label": LABEL_COLUMNS,
        "probability": probs
    })

    return result.sort_values("probability", ascending=False)

example_text = "I am so happy today"
display(predict_emotions(example_text))

## 19. What to submit / keep
Important saved outputs inside `OUTPUT_DIR`:
- `best_model/`
- `best_thresholds.json`
- `dev_metrics_default.json`
- `dev_metrics_tuned.json`
- `dev_predictions_with_probs.csv`

